# SegFormer-B0 defect segmentation (Kaggle, T4 x2)

**Settings:** Accelerator = GPU T4 x2, Internet = ON, Persistence = Files only.
**Inputs to add:** the `defect-scripts` dataset (train_segformer.py + prepare_data.py) and any Kaggle crack datasets (e.g. via *Add Input* from the DeepCrack / Crack500 notebooks' data sources).

In [ ]:
!nvidia-smi
!ls /kaggle/input

In [ ]:
!pip install -q transformers albumentations

## CrackSeg9k Data Path
Kaggle auto-extracted the dataset during upload. Pointing directly to the input directory.

In [ ]:
print("Dataset directory structure:")
!find /kaggle/input/datasets/himonklolo/crackseg-dataset -maxdepth 4 -type d | head -30

## Config + data prep

In [ ]:
# Auto-detect scans the input directory directly to find matching Images and Masks folders
AUTO = {
    "crackseg9k": "/kaggle/input/datasets/himonklolo/crackseg-dataset",
}

# Scripts and output paths
SCRIPTS = "/kaggle/input/datasets/himonklolo/defect-scripts"
DATA, RUN = "/kaggle/working/data", "/kaggle/working/runs/b0_v1"

# Set for 3-epoch trial run
EPOCHS, BS, LR = 3, 16, 6e-5   # BS is PER GPU (T4x2 -> effective batch size = 32)

In [ ]:
import shutil, subprocess
for f in ("train_segformer.py", "prepare_data.py"):
    shutil.copy(f"{SCRIPTS}/{f}", f"/kaggle/working/{f}")
os.chdir("/kaggle/working")
auto = sum([["--auto", f"{n}={p}"] for n, p in AUTO.items()], [])
subprocess.run(["python", "prepare_data.py", "--dry_run", *auto], check=True)   # CHECK this output: pairs must look right


In [ ]:
if not os.path.isdir(f"{DATA}/train/images"):
    subprocess.run(["python", "prepare_data.py", "--out", DATA, "--max_side", "1024", "--val_frac", "0.1", *auto], check=True)
else:
    print("data already prepared")


## Train (DDP on both T4s)

In [ ]:
# Resumes automatically if last.pt exists (after the 12h limit: attach previous output as input, copy last.pt into RUN)
# workers=2 per process because Kaggle only gives ~4 CPU cores.
resume = f"--resume {RUN}/last.pt" if os.path.exists(f"{RUN}/last.pt") else ""
!torchrun --nproc_per_node=2 train_segformer.py --data_root {DATA} --out_dir {RUN} --img_size 512 \
    --batch_size {BS} --epochs {EPOCHS} --lr {LR} --workers 2 {resume}
# If NCCL hangs/errors on init, rerun with:  !NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 torchrun ...


In [ ]:
import json
m = json.load(open(f"{RUN}/metrics.json"))
best = max(m, key=lambda x: x["mIoU"])
print("best epoch:", best["epoch"], {k: round(v, 4) for k, v in best.items() if k != "epoch"})
shutil.make_archive("/kaggle/working/segformer_b0_best", "zip", f"{RUN}/best")   # download from Output tab
